# PrivateLocalAgent

1. **Kernel → Restart Kernel**  
2. 只运行下面 **这一个** 代码单元格（不要再跑旧的第二格）

In [ ]:
import os, sys, subprocess
from pathlib import Path

def log(msg):
    print(msg, flush=True)

ROOT = Path("/workspace/Radeon-hackathon-2026-07")
if not (ROOT / "src" / "config.py").is_file():
    here = Path.cwd()
    ROOT = here.parent if here.name == "notebooks" else here
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# Make `import src` work even if cwd is notebooks/
nb_link = ROOT / "notebooks" / "src"
if not nb_link.exists():
    try:
        nb_link.symlink_to(ROOT / "src", target_is_directory=True)
    except Exception:
        pass

log(f"ROOT={ROOT}")

persist = Path("/workspace/persistence")
if not persist.is_dir():
    persist = Path("/persistent")
if persist.is_dir():
    os.environ.setdefault("PLA_DATA_ROOT", str(persist / "PrivateLocalAgent"))
    os.environ.setdefault("HF_HOME", str(persist / "huggingface"))
    Path(os.environ["PLA_DATA_ROOT"]).mkdir(parents=True, exist_ok=True)
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("USE_ROCM_AITER_ROPE_BACKEND", "0")

log("[0] pip...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "chromadb", "sentence-transformers", "pypdf", "pyyaml",
    "python-dotenv", "pydantic", "openai", "ipywidgets",
    "transformers", "accelerate", "safetensors", "sentencepiece",
    "Pillow", "rapidocr-onnxruntime",
])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])

from src.agent.agent import PrivateAgent
from src.agent.multi_agent import MultiAgentOrchestrator
from src.agent.tools import ToolRegistry
from src.apps.judge_script import ensure_judge_ocr_image
from src.app.notebook_visual import launch_notebook_visual
from src.config import load_settings
from src.llm.backend import build_llm
from src.memory.memory import SessionMemory
from src.privacy.audit import AuditTrail
from src.rag.store import VectorStore
from src.skills import SkillRegistry

settings = load_settings()
upload_dir = settings.resolve(settings.paths.upload_dir)
ensure_judge_ocr_image(upload_dir)

log("[1] KB...")
store = VectorStore(settings)
n = store.ensure_sample_docs(settings.resolve(settings.paths.sample_docs))
log(f"    ingested={n} chunks={store.count()}")

memory = SessionMemory(settings.resolve(settings.agent.memory_path))
skills = SkillRegistry(settings.resolve(settings.paths.generated_projects))
tools = ToolRegistry(store, memory, upload_dir, skill_registry=skills)
audit = AuditTrail(settings.resolve("data/memory/audit.jsonl"))

log("[2] load LLM...")
llm = build_llm(settings.llm)
agent = PrivateAgent(llm, tools, memory, settings.agent.max_steps, audit=audit)
orch = MultiAgentOrchestrator(agent, tools)
log("ready")

ui = launch_notebook_visual(orch, default_mode="chat")